In [1]:
import pandas as pd
import numpy as np
import re
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
INPUT_STRUCTURED = "Information_Systems_structured.csv"

# If master file is one folder above notebooks folder:
MASTER_SCHEMA_PATH = "../All_Journals_CLEANED.csv"

OUTPUT_FINAL = "Information_Systems_final_for_powerBI.csv"
OUTPUT_REVIEW = "Information_Systems_university_review.csv"

In [3]:
print("Current working directory:", os.getcwd())
print("Input exists:", os.path.exists(INPUT_STRUCTURED))
print("Master exists:", os.path.exists(MASTER_SCHEMA_PATH))

Current working directory: /Users/keerthisagi/Documents/Journals/Journal_of_Information_Systems
Input exists: True
Master exists: True


In [4]:
df = pd.read_csv(INPUT_STRUCTURED)
print("Rows:", len(df))
df.head()

Rows: 4680


,URL,Journal_Title,Article_Title,Volume_Issue,Month_Year,Abstract,Keywords,Author_name,Author_email,Author_Address,Author_Email_Extracted,Author_University,Author_Country,Author_Department,Standardized_Author
0,https://www.sciencedirect.com/science/article/...,Information Systems,Efficient data structures for fast and low-cos...,Volume 139,July 2026,NaN,"['Rule mining', 'Logic rule', 'Data structure'...",Ruoyu Wang,ruoyu.wang2@unsw.edu.au,"University of New South Wales, Sydney, 2032, U...",NaN,University of New South Wales,Australia,NaN,Ruoyu Wang
1,https://www.sciencedirect.com/science/article/...,Information Systems,Efficient data structures for fast and low-cos...,Volume 139,July 2026,NaN,"['Rule mining', 'Logic rule', 'Data structure'...",Raymond Wong,ray.wong@unsw.edu.au,"University of New South Wales, Sydney, 2032, U...",NaN,University of New South Wales,Australia,NaN,Raymond Wong
2,https://www.sciencedirect.com/science/article/...,Information Systems,Efficient data structures for fast and low-cos...,Volume 139,July 2026,NaN,"['Rule mining', 'Logic rule', 'Data structure'...",Daniel Sun,danielwsun@gmail.com,"Newcastle University, Tyne and Wear, England, ...",NaN,Newcastle University,United Kingdom,NaN,Daniel Sun
3,https://www.sciencedirect.com/science/article/...,Information Systems,Measuring the decentralisation of DeFi develop...,Volume 139,July 2026,NaN,"['DeFi', 'Contributor structure', 'Decentralis...",Giuseppe Destefanis,g.destefanis@ucl.ac.uk,"Department of Computer Science, University Col...",NaN,University College London,United Kingdom,Department of Computer Science,Giuseppe Destefanis
4,https://www.sciencedirect.com/science/article/...,Information Systems,Measuring the decentralisation of DeFi develop...,Volume 139,July 2026,NaN,"['DeFi', 'Contributor structure', 'Decentralis...",Jiahua Xu,NaN,"Department of Computer Science, University Col...",NaN,University College London,United Kingdom,Department of Computer Science,Jiahua Xu


In [5]:
def clean_spaces(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if not s:
        return np.nan
    s = re.sub(r"\s+", " ", s)
    return s

def standardize_title(x):
    # Do NOT force title-case; just normalize spacing (more academically safe)
    return clean_spaces(x)

def standardize_author(x):
    s = clean_spaces(x)
    if pd.isna(s):
        return np.nan
    # Title-case names for consistency
    return s.title()

def standardize_university(x):
    return clean_spaces(x)

In [6]:
# Ensure these exist even if empty
needed = [
    "URL", "Journal_Title", "Standardized_Title", "month_year", "Abstract", "Keywords",
    "Author_Name", "Standardized_Author", "Author_University", "Standardized_University",
    "Author_State", "Author_Country", "Author_Address"
]

for c in needed:
    if c not in df.columns:
        df[c] = np.nan

In [7]:
# Fill Standardized_Title from other likely title columns if missing/empty
if df["Standardized_Title"].isna().all() or (df["Standardized_Title"].astype(str).str.strip() == "").all():
    if "Title" in df.columns:
        df["Standardized_Title"] = df["Title"]
    elif "Article_Title" in df.columns:
        df["Standardized_Title"] = df["Article_Title"]

df["Standardized_Title"] = df["Standardized_Title"].apply(standardize_title)

In [8]:
# If your notebook 2 used Author_name (lowercase), bring it in
if df["Author_Name"].isna().all() and "Author_name" in df.columns:
    df["Author_Name"] = df["Author_name"]

# Build Standardized_Author from Author_Name if missing
df["Standardized_Author"] = df["Standardized_Author"].fillna(df["Author_Name"])
df["Standardized_Author"] = df["Standardized_Author"].apply(standardize_author)

# Keep Author_Name populated for Power BI friendliness
df["Author_Name"] = df["Author_Name"].fillna(df["Standardized_Author"])
df["Author_Name"] = df["Author_Name"].apply(clean_spaces)

In [9]:
# Normalize Author_University
df["Author_University"] = df["Author_University"].apply(standardize_university)

# Collect unique non-empty universities
unique_unis = sorted([u for u in df["Author_University"].dropna().unique() if str(u).strip() != ""])
print("Unique universities:", len(unique_unis))

Unique universities: 949


In [10]:
threshold = 0.85
mapping = {}

if len(unique_unis) > 1:
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), lowercase=True)
    X = vectorizer.fit_transform(unique_unis)
    sim = cosine_similarity(X)

    for i, uni_i in enumerate(unique_unis):
        # Find best match index (excluding itself)
        scores = sim[i].copy()
        scores[i] = 0.0
        j = int(scores.argmax())
        best_score = float(scores[j])

        if best_score >= threshold:
            uni_j = unique_unis[j]
            # Canonical = longer string (usually full university name)
            canonical = uni_i if len(uni_i) >= len(uni_j) else uni_j
            duplicate = uni_j if canonical == uni_i else uni_i
            mapping[duplicate] = canonical

print("Mappings created:", len(mapping))

Mappings created: 54


In [11]:
df["University_Canonical"] = df["Author_University"].replace(mapping)

# Standardized_University should be the canonical one
df["Standardized_University"] = df["University_Canonical"].fillna(df["Author_University"])
df["Standardized_University"] = df["Standardized_University"].apply(standardize_university)

In [12]:
review_df = pd.DataFrame(
    [{"Duplicate_Name": k, "Canonical_Name": v} for k, v in mapping.items()]
).sort_values(["Canonical_Name", "Duplicate_Name"])

review_df.to_csv(OUTPUT_REVIEW, index=False)
print("Saved review file:", OUTPUT_REVIEW)

Saved review file: Information_Systems_university_review.csv


In [13]:
df = df.replace(["nan", "NaN", "None", "N/A", ""], np.nan)

In [14]:
master = pd.read_csv(MASTER_SCHEMA_PATH)
master_cols = master.columns.tolist()

# Add missing master columns
for c in master_cols:
    if c not in df.columns:
        df[c] = np.nan

# Keep only master columns in exact order
df_final = df[master_cols].copy()

print("Columns match master exactly:", list(df_final.columns) == master_cols)
print("Final rows:", len(df_final))

Columns match master exactly: True
Final rows: 4680


In [15]:
df_final.to_csv(OUTPUT_FINAL, index=False)
print("Saved final Power BI file:", OUTPUT_FINAL)

Saved final Power BI file: Information_Systems_final_for_powerBI.csv
